# 🔬 Notebook 3: Distributed Lock Manager — Deep Dive: Fencing end-to-end

## 🛠️ Setup

```bash
cd 06-system-designs/distributed-lock-manager
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we'll build

A toy `LockManager` + a `FencedResource` (the "protected thing", e.g. a database row),
and then we'll reproduce the classic **GC pause → split brain** scenario twice:

1. **Bad**: lock with TTL only, resource trusts its caller → data corrupted.
2. **Best**: same lock but now the resource checks the fencing token → corruption prevented.


In [ ]:
# Toy lock manager (same shape as Notebook 2's InMemoryLockService, plus renew()).
import time, threading
from dataclasses import dataclass

@dataclass
class LockEntry:
    owner: str
    token: int
    expires_at: float

class LockManager:
    def __init__(self):
        self._locks: dict[str, LockEntry] = {}
        self._next_token = 0
        self._mu = threading.Lock()

    def _now(self): return time.time()

    def acquire(self, name, owner, ttl_s):
        with self._mu:
            e = self._locks.get(name)
            if e and e.expires_at > self._now() and e.owner != owner:
                return None  # held by someone else
            self._next_token += 1
            e = LockEntry(owner, self._next_token, self._now() + ttl_s)
            self._locks[name] = e
            return {"token": e.token, "expires_at": e.expires_at}

    def renew(self, name, owner, token, ttl_s):
        with self._mu:
            e = self._locks.get(name)
            if not e or e.owner != owner or e.token != token: return None
            if e.expires_at < self._now(): return None
            e.expires_at = self._now() + ttl_s
            return {"expires_at": e.expires_at}

    def release(self, name, owner, token):
        with self._mu:
            e = self._locks.get(name)
            if e and e.owner == owner and e.token == token:
                del self._locks[name]; return True
            return False

lm = LockManager()
a1 = lm.acquire("nightly-job", "worker-A", ttl_s=1.0)
print("A acquires:", a1)
print("B denied  :", lm.acquire("nightly-job", "worker-B", ttl_s=1.0))
time.sleep(1.1)  # let A's lease expire
a2 = lm.acquire("nightly-job", "worker-B", ttl_s=1.0)
print("B after expiry:", a2, "← note the token is strictly greater")


## 🚫 Bad: TTL-only lock, resource trusts the caller

Simulate a GC pause: worker A acquires, then "pauses" (sleeps longer than the TTL).
Worker B notices A's lease expired and takes over. A wakes up **still believing it
holds the lock** and writes to the shared resource. Without a fencing check, that
write is accepted and **corrupts the data**.


In [ ]:
class UncheckedResource:
    """A resource that trusts anyone who calls it (the 'bad' design)."""
    def __init__(self): self.state = []
    def write(self, who, data):
        self.state.append((who, data))
        print(f"  ACCEPT (unchecked): {who} wrote {data!r}")

lm = LockManager()
res = UncheckedResource()

# A acquires with a short 0.5s TTL, then goes into a long "GC pause"
a = lm.acquire("job", "A", ttl_s=0.5)
print("A acquired token=", a["token"])
print("A starts pause (1.2s) — imagine a stop-the-world GC")
time.sleep(1.2)

# While A was paused, B's heartbeat noticed expiry and B acquired
b = lm.acquire("job", "B", ttl_s=5.0)
print("B acquired token=", b["token"])
res.write("B", "correct-result")

# A wakes up, still believes it holds the lock, and writes
print("A wakes up and writes (DANGER):")
res.write("A", "stale-result-from-before-pause")

print("final state →", res.state)
print("❌ corrupted: B's correct write is followed by A's stale write")


## 🏆 Best: resource checks the fencing token

Now the resource remembers the **highest token** it has ever seen and rejects any
write carrying a lower token. A's stale write from before the pause can't win
because its token is smaller than B's.


In [ ]:
class FencedResource:
    """A resource that only accepts writes with a monotonically non-decreasing token."""
    def __init__(self):
        self.last_token = 0
        self.state = []
    def write(self, token, who, data):
        if token < self.last_token:
            print(f"  REJECT: token {token} from {who} < last_seen {self.last_token}")
            return False
        self.last_token = token
        self.state.append((token, who, data))
        print(f"  ACCEPT: token {token} from {who} wrote {data!r}")
        return True

lm = LockManager()
res = FencedResource()

a = lm.acquire("job", "A", ttl_s=0.5)
print("A acquired token=", a["token"])
time.sleep(1.2)  # GC pause

b = lm.acquire("job", "B", ttl_s=5.0)
print("B acquired token=", b["token"])
res.write(b["token"], "B", "correct-result")

# A wakes up and tries to use its OLD token → rejected
print("A wakes up and tries to write with its stale token:")
res.write(a["token"], "A", "stale-result-from-before-pause")

print("final state →", res.state)
print("✅ safe: A's stale write was rejected because a_token < b_token")


## Lease renewal — how long-running jobs stay safe

Picking a TTL is a tradeoff:
- **Too long** → if a holder crashes, everyone waits forever.
- **Too short** → a slow-but-healthy job loses the lock mid-work.

The standard fix is a **heartbeat**: pick a short TTL (say 10s) and renew every
few seconds. If the holder dies, the lease expires quickly; if it's alive, it
keeps extending.


In [ ]:
# Heartbeat demo: A holds a 0.4s lease and renews every 0.15s for ~1 second.
# B, which just polls acquire(), should NEVER get in.
import threading, time

lm = LockManager()
stop = threading.Event()

def heartbeat(name, owner, token):
    while not stop.is_set():
        ok = lm.renew(name, owner, token, ttl_s=0.4)
        if ok is None:
            print("  heartbeat: lost the lease!")
            return
        time.sleep(0.15)

a = lm.acquire("job", "A", ttl_s=0.4)
hb = threading.Thread(target=heartbeat, args=("job", "A", a["token"]))
hb.start()

for i in range(6):
    time.sleep(0.15)
    got = lm.acquire("job", "B", ttl_s=0.4)
    print(f"t={i*0.15:.2f}s  B got lock? {bool(got)}")

stop.set(); hb.join()
print("A finishes and releases:", lm.release("job", "A", a["token"]))


## Redis `SET NX PX` — the pragmatic minimum

If you don't need consensus-level guarantees, Redis is the common choice.

```python
# Acquire (atomic on a single Redis):
SET lock:nightly-job worker-A NX PX 30000

# Release must check ownership — use a Lua script for atomicity:
EVAL "if redis.call('get', KEYS[1]) == ARGV[1] \
      then return redis.call('del', KEYS[1]) \
      else return 0 end"  1  lock:nightly-job  worker-A

# Fencing token: INCR a separate key in the SAME Lua script so acquire returns (lock, token).
```

### When Redis is not enough
- Redis master fails right after `SET NX` → replica without the key is promoted →
  two clients can each hold the lock. **Redlock** tries to mitigate by requiring a
  majority of independent nodes; still vulnerable to clock skew
  ([Kleppmann 2016](https://martin.kleppmann.com/2016/02/08/how-to-do-distributed-locking.html)).
- For correctness-critical locks (leader election, schema migration, payment
  reconciliation), use a **consensus** system (etcd, ZooKeeper) and still emit a
  fencing token.


## Real-world examples

| System                    | What they lock                                 | Backend          | Notes |
|---------------------------|------------------------------------------------|------------------|-------|
| Kubernetes controllers    | "I am the active controller for this resource" | etcd lease       | Leader election via `coordination.k8s.io/Lease`. |
| Kafka (pre-KRaft)         | "I am the controller broker"                  | ZooKeeper znode  | Ephemeral node; dies with the session. |
| Patroni (Postgres HA)     | "I am the primary"                            | etcd / Consul    | Timeouts + TTLs picked carefully. |
| Google Chubby             | Any coarse-grained lock                        | Paxos            | The paper that started the whole field. |
| Redis-backed job runners  | "Only one worker runs this cron"              | Redis `SET NX`   | Fast and good enough for most cron-like workloads. |
| HBase / Bigtable          | Region / tablet ownership                      | ZK / Chubby      | Combined with fencing via region epoch. |

### Checklist before shipping a distributed lock
1. Does the **protected resource** check a fencing token? (If not, TTLs are a lie.)
2. Is the **TTL shorter than your worst-case detection time** for a dead owner?
3. Do you have **heartbeat/renew** for long jobs?
4. Is `release` **idempotent** and **owner-checked**?
5. What happens if the **lock backend** itself fails over? (Split-brain budget.)
6. Are acquire/release paths on your **critical latency path**? (Caching, pipelining.)
